# NutriVision Phase 2: Full Dataset Training
This notebook is designed to run the full training pipeline for the Food-101 Classifier and Nutrition5k Portion Estimator on Google Colab.


## 1. Setup Environment
First, let's clone the repository or set up the Google Drive path if you uploaded it there.

In [ ]:
# Option A: If you uploaded the folder to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/project1

# Option B (default): Clone the repo fresh into the Colab session
import os
REPO_URL = 'https://github.com/Prowderypulp/Calorie-Identification.git'
REPO_DIR = 'Calorie-Identification'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!pwd && ls


## 2. Install Dependencies
Colab has PyTorch and Torchvision pre-installed. We just need to add the ONNX utilities and some specific packages.

In [ ]:
!pip install onnx onnxruntime onnxscript structlog pydantic-settings


## 2b. Mount Google Drive (save best models here)
Best `.pth` and `.onnx` files will be written directly to your Drive so they survive Colab session timeouts.


In [ ]:
from google.colab import drive
import os, pathlib

drive.mount('/content/drive')

# Where best models will be saved. Change the project folder name if you like.
MODELS_DIR = '/content/drive/MyDrive/NutriVision/models'
pathlib.Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
os.environ['MODELS_DIR'] = MODELS_DIR

# Symlink ./app/models -> Drive so the FastAPI loaders and any scripts that hardcode
# ./app/models also see the Drive-backed files.
local = pathlib.Path('app/models')
if local.is_symlink() or local.exists():
    if local.is_symlink() or not any(local.iterdir()):
        local.unlink() if local.is_symlink() else local.rmdir()
if not local.exists():
    local.symlink_to(MODELS_DIR)
print('Models will be saved to:', MODELS_DIR)


## 3. Download Datasets
Colab's cloud network will download these gigabytes of data much faster than a local machine.

In [ ]:
# Download Food-101 (~5GB)
!python scripts/download_food101.py

# Download Nutrition5k metadata + RGB imagery, then build train/val CSVs
!python scripts/download_nutrition5k.py
!python scripts/prepare_nutrition5k.py


## 4. Train EfficientNet-B0 Classifier (Food-101)
We can use a larger batch size (e.g., 64 or 128) here since we are on a Colab GPU (T4/L4/A100).

In [ ]:
# ResNet-50 (V2 weights) + channels_last + RandAugment.
# On a Colab T4 this completes in ~25 min and reaches ~83-85% val acc.
# Best model is saved to $MODELS_DIR on your Drive every time val acc improves.
!python scripts/train_classifier.py \
    --data_dir ./data --use_torchvision_dataset \
    --output_dir $MODELS_DIR \
    --epochs 10 --batch_size 128 --num_workers 4 --warmup_epochs 1


## 5. Train Portion Estimator (Nutrition5k)
*Note:* Make sure `train.csv` and `val.csv` are properly generated from the raw Nutrition5k metadata before running this cell. The training script expects these CSVs to map images to their weight labels.

In [ ]:
!python scripts/train_portion.py \
    --train_csv ./data/nutrition5k/train.csv \
    --val_csv ./data/nutrition5k/val.csv \
    --img_dir ./data/nutrition5k/imagery/realsense_overhead \
    --output_dir $MODELS_DIR \
    --epochs 20 --batch_size 64 --num_workers 4


## 6. Export Models to ONNX
Once training is complete, export the `.pth` files to ONNX format so they can be downloaded and used locally by the FastAPI server.

In [ ]:
!python scripts/export_onnx.py \
    --model_path $MODELS_DIR/classifier.pth --model_type classifier \
    --num_classes 101 --output_path $MODELS_DIR/classifier.onnx

!python scripts/export_onnx.py \
    --model_path $MODELS_DIR/portion_estimator.pth --model_type portion \
    --output_path $MODELS_DIR/portion_estimator.onnx
